# Day 25: Text Chunking Strategies

## Core Theory: Why We Chunk and How We Do It

When building Retrieval-Augmented Generation (RAG) applications, you cannot pass an entire 100-page document into a language model's context window. Even if you could, providing too much irrelevant context degrades the model's performance and significantly increases latency and cost. This is where **chunking** comes in.

Chunking is the process of breaking down large documents into smaller, semantically meaningful pieces (chunks). These chunks are then embedded and stored in a vector database (like Qdrant) for retrieval. The quality of your RAG pipeline heavily depends on your chunking strategy. If chunks are too small, they lose context. If they are too large, they introduce noise and exceed token limits.

Today, we will compare two foundational chunking strategies:
1.  **Token-Based Chunking:** Splits text based on the raw number of tokens. It guarantees that the chunk will fit neatly into a model's token limits, but it completely ignores the semantic structure of the text (e.g., cutting a sentence or a word in half).
2.  **Recursive Character Text Splitting:** A smarter approach. It tries to split text on natural boundaries (like double newlines `\n\n`, single newlines `\n`, spaces ` `, and finally characters `""`) to keep related sentences and paragraphs together. It falls back to smaller units only when a block of text exceeds the allowed size.


### AI Security Implications
1. **PII Leakage in Chunks:** When splitting text, Personally Identifiable Information (PII) might be split across two chunks, making simple regex or NER redaction fail if applied post-chunking. Always redact PII *before* chunking.
2. **Prompt Injection Risk:** If user-generated content is chunked and retrieved alongside trusted documents, an attacker might embed partial prompt injections that combine during the RAG assembly phase. Implement strict isolation and context tagging.
3. **Fallback Mechanisms:** If the splitter fails (e.g., encounters an unexpected encoding or format), implement a fallback to a safer, simpler chunking method (like basic token splitting) to ensure the pipeline doesn't crash.


## Common Pitfalls in Production

1.  **Orphaned Context:** Using a basic token splitter often breaks paragraphs mid-sentence. When retrieved, the model lacks the full context to answer accurately.
2.  **Zero Overlap:** Not including an overlap between chunks can lead to missing context at chunk boundaries. Always configure a reasonable `chunk_overlap` (e.g., 10-20% of the chunk size).
3.  **Ignoring the Source Structure:** Treating all text (code, markdown, raw text) the same. `RecursiveCharacterTextSplitter` is great for general text, but for code or highly structured Markdown, specialized splitters are better.
4.  **Mismatched Tokenizers:** Tokenizing with a splitter that uses a different tokenizer than your LLM. Always match the tokenizer (e.g., `tiktoken` with `cl100k_base` for OpenAI models) when strict token limits are required.

## Code Implementation

Let's explore a tiered progression from Basic to Advanced implementations.

### Basic: Core Concept Isolation
This example demonstrates the raw difference between token-based chunking and recursive character chunking with minimal boilerplate.


In [1]:
from typing import List
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter
import tiktoken

def demonstrate_chunking() -> None:
    """
    Demonstrates the difference between token-based chunking and 
    recursive character chunking using LangChain.
    """
    
    sample_text = (
        "Artificial Intelligence (AI) is transforming the world.\n\n"
        "Machine learning models, particularly Large Language Models (LLMs), "
        "have shown remarkable capabilities in natural language understanding and generation.\n\n"
        "However, these models have a limited context window. Therefore, when building "
        "Retrieval-Augmented Generation (RAG) systems, we must carefully chunk our data. "
        "If we just split blindly, we might cut a sentence right in the middle, losing the "
        "core meaning of the text."
    )

    print("--- Original Text Length ---")
    print(f"Characters: {len(sample_text)}")
    
    # 1. Token-Based Splitting
    # This splits strictly by the number of tokens, ignoring punctuation or paragraphs.
    token_splitter = TokenTextSplitter(
        chunk_size=20,
        chunk_overlap=5
    )
    
    token_chunks: List[str] = token_splitter.split_text(sample_text)
    
    print("\n--- Token-Based Chunking (chunk_size=20) ---")
    for i, chunk in enumerate(token_chunks):
        print(f"Chunk {i+1}: {repr(chunk)}")
        
    # 2. Recursive Character Splitting
    # This attempts to split at logical boundaries (\n\n, \n, space, empty string).
    # It prioritizes keeping paragraphs and sentences intact.
    recursive_splitter = RecursiveCharacterTextSplitter(
        chunk_size=150,
        chunk_overlap=20,
        separators=["\n\n", "\n", " ", ""]
    )
    
    recursive_chunks: List[str] = recursive_splitter.split_text(sample_text)
    
    print("\n--- Recursive Character Chunking (chunk_size=150) ---")
    for i, chunk in enumerate(recursive_chunks):
        print(f"Chunk {i+1}: {repr(chunk)}")

if __name__ == "__main__":
    demonstrate_chunking()


--- Original Text Length ---
Characters: 476



--- Token-Based Chunking (chunk_size=20) ---
Chunk 1: 'Artificial Intelligence (AI) is transforming the world.\n\nMachine learning models, particularly Large Language'
Chunk 2: ' models, particularly Large Language Models (LLMs), have shown remarkable capabilities in natural language understanding and generation'
Chunk 3: ' natural language understanding and generation.\n\nHowever, these models have a limited context window. Therefore,'
Chunk 4: ' context window. Therefore, when building Retrieval-Augmented Generation (RAG) systems, we'
Chunk 5: 'RAG) systems, we must carefully chunk our data. If we just split blindly, we might cut'
Chunk 6: ' blindly, we might cut a sentence right in the middle, losing the core meaning of the text.'

--- Recursive Character Chunking (chunk_size=150) ---
Chunk 1: 'Artificial Intelligence (AI) is transforming the world.'
Chunk 2: 'Machine learning models, particularly Large Language Models (LLMs), have shown remarkable capabilities in natural language 

### Medium: Clean OOP and State Management
Here we encapsulate the chunking logic into a clean class structure, managing state such as the selected tokenizer and default parameters.


In [2]:
from typing import List
from langchain_text_splitters import RecursiveCharacterTextSplitter

class DocumentProcessor:
    """
    Encapsulates text splitting logic for a document processing pipeline.
    """
    def __init__(self, chunk_size: int = 200, chunk_overlap: int = 20):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        # Initialize the splitter as part of the object's state
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )
        
    def process_text(self, text: str) -> List[str]:
        """
        Processes the input text and returns a list of chunks.
        """
        if not text.strip():
            return []
        return self._splitter.split_text(text)

if __name__ == "__main__":
    processor = DocumentProcessor(chunk_size=100, chunk_overlap=10)
    sample = "This is a medium complexity example.\n\nIt uses classes for better organization."
    chunks = processor.process_text(sample)
    for i, c in enumerate(chunks):
        print(f"Chunk {i+1}: {c}")


Chunk 1: This is a medium complexity example.

It uses classes for better organization.


### Advanced: Production-Grade Implementation
This version includes strict type hinting, comprehensive error handling, fallback mechanisms, and AI security considerations (simulated PII redaction).


In [3]:
import logging
import re
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ChunkingConfig(BaseModel):
    """Strict configuration validation for chunking parameters."""
    chunk_size: int = Field(gt=0, le=8192, description="Must be a positive integer within model limits.")
    chunk_overlap: int = Field(ge=0, description="Cannot be negative.")
    
class SecureDocumentChunker:
    """
    Production-grade chunker with fallback mechanisms and PII redaction.
    """
    def __init__(self, config: ChunkingConfig):
        self.config = config
        try:
            self.primary_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                chunk_size=self.config.chunk_size,
                chunk_overlap=self.config.chunk_overlap,
                model_name="gpt-4"
            )
            self.fallback_splitter = TokenTextSplitter(
                chunk_size=self.config.chunk_size,
                chunk_overlap=self.config.chunk_overlap
            )
        except Exception as e:
            logger.error(f"Failed to initialize splitters: {e}")
            raise
            
    def _redact_pii(self, text: str) -> str:
        """
        Redacts emails as a basic PII protection step before chunking.
        """
        # Robust email regex
        email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
        return re.sub(email_pattern, '[REDACTED_EMAIL]', text)

    def chunk_document(self, text: str) -> List[str]:
        """
        Safely chunks a document, applying security filters and fallback logic.
        """
        if not text:
            logger.warning("Received empty text for chunking.")
            return []
            
        secure_text = self._redact_pii(text)
        
        try:
            logger.info("Attempting primary recursive chunking.")
            return self.primary_splitter.split_text(secure_text)
        except Exception as e:
            logger.error(f"Primary chunker failed: {e}. Attempting fallback.")
            try:
                return self.fallback_splitter.split_text(secure_text)
            except Exception as fallback_e:
                logger.critical(f"Fallback chunker also failed: {fallback_e}")
                raise RuntimeError("All chunking strategies failed.") from fallback_e

if __name__ == "__main__":
    try:
        config = ChunkingConfig(chunk_size=100, chunk_overlap=10)
        chunker = SecureDocumentChunker(config)
        
        test_doc = "Contact us at admin@example.com for more info. " * 20
        chunks = chunker.chunk_document(test_doc)
        
        logger.info(f"Successfully created {len(chunks)} chunks.")
        print(f"Sample chunk: {chunks[0]}")
    except ValidationError as ve:
        logger.error(f"Configuration Error: {ve}")
    except Exception as ex:
        logger.error(f"Execution Error: {ex}")


INFO:__main__:Attempting primary recursive chunking.


INFO:__main__:Successfully created 3 chunks.


Sample chunk: Contact us at [REDACTED_EMAIL] for more info. Contact us at [REDACTED_EMAIL] for more info. Contact us at [REDACTED_EMAIL] for more info. Contact us at [REDACTED_EMAIL] for more info. Contact us at [REDACTED_EMAIL] for more info. Contact us at [REDACTED_EMAIL] for more info. Contact us at [REDACTED_EMAIL] for more info. Contact us


## Practical Lab: Implementing a Hybrid Splitter Function

**Your Task:**
Write a function `chunk_document(text: str, max_tokens: int) -> List[str]` that behaves like a hybrid approach:
1. It should use `RecursiveCharacterTextSplitter` but it should evaluate chunk sizes based on *token count*, not character count.
2. LangChain's `RecursiveCharacterTextSplitter` provides a class method `from_tiktoken_encoder` to achieve this.
3. Configure it to use a `chunk_size` of 50 tokens and a `chunk_overlap` of 10 tokens.
4. Test it on a long string of your choice to verify it respects sentence boundaries as much as possible while strictly adhering to the token limits.

**Deliverable:**
Record a brief (2-3 minute) async video walkthrough of your design decisions, explaining how you handled token limits and any edge cases you encountered.


In [4]:
from typing import List
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_document(text: str, max_tokens: int) -> List[str]:
    """
    Splits text recursively while measuring chunk length in tokens instead of characters.
    
    Args:
        text (str): The input document to be chunked.
        max_tokens (int): The maximum number of tokens allowed per chunk.
        
    Returns:
        List[str]: A list of text chunks.
    """
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=max_tokens,
        chunk_overlap=10,
        model_name="gpt-4",
    )
    
    return splitter.split_text(text)

test_text = (
    "The process of chunking is absolutely vital for the success of Retrieval-Augmented Generation (RAG). "
    "If you chunk poorly, the LLM will hallucinate or complain about lack of context. "
    "If you chunk well, the system feels magical. "
    "\n\nLet's ensure we are counting tokens correctly while keeping semantic meaning intact. "
    "Recursive splitting helps us achieve this by breaking at paragraphs, sentences, and words in that order."
)

print("--- Hybrid Splitting (Max 50 Tokens) ---")
chunks = chunk_document(test_text, max_tokens=50)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {repr(chunk)}")


--- Hybrid Splitting (Max 50 Tokens) ---
Chunk 1: 'The process of chunking is absolutely vital for the success of Retrieval-Augmented Generation (RAG). If you chunk poorly, the LLM will hallucinate or complain about lack of context. If you chunk well, the system feels magical.'
Chunk 2: "Let's ensure we are counting tokens correctly while keeping semantic meaning intact. Recursive splitting helps us achieve this by breaking at paragraphs, sentences, and words in that order."


## Reference Links

- [LangChain Text Splitters Documentation](https://python.langchain.com/docs/how_to/#text-splitters)
- [OpenAI Tokenizer (tiktoken) Guide](https://github.com/openai/tiktoken)
- [Qdrant Concepts: Chunking and Embeddings](https://qdrant.tech/articles/what-is-embeddings/)
